# Mask2Former on the smart-crops datasets (cabbage / purple kale / yellow onion)

Instance segmentation with **Mask2Former** (detectron2), trained separately on each of the
three crop datasets, so the numbers line up with the YOLO26s-seg runs.

**Kaggle settings required**
- Accelerator: **GPU T4 x2** or **GPU P100** (P100 is fine, only 1 GPU is used by default)
- **Internet: ON** (needed to pip-install detectron2 and download pretrained weights)
- Attach the `smart-crops` dataset

**What this notebook does**
1. Builds detectron2 + Mask2Former + the MSDeformAttn CUDA kernel (with the patches needed for torch 2.x).
2. Converts the YOLO-segmentation labels (`labels/*.txt` polygons) into COCO instance JSON.
3. Registers each crop dataset with detectron2 and fine-tunes `maskformer2_R50_bs16_50ep` from COCO weights.
4. Evaluates on the `test` split with COCO mask AP (`segm AP`, `AP50`, `AP75`) - directly comparable to `yolo segment val`.

**Timing:** ~15-20 min for the build, then ~1.5-3 h per dataset at the default 50 epochs.
Setup is a single session cost; if you only have time for one dataset per session, run the
setup cells then just the one train/eval pair you want (same pattern as the YOLO notebook).

In [ ]:
!nvidia-smi
import sys, torch, numpy, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
print("numpy   ", numpy.__version__)
print("gpus    ", torch.cuda.device_count())

## 1. Build the environment

detectron2 and the MSDeformAttn kernel are compiled from source. This takes ~15 min and
must be re-run each session (Kaggle does not persist `site-packages`).

If you want to skip the rebuild in later sessions: after this cell succeeds once, run the
"save the build" cell at the bottom, publish `/kaggle/working/build_cache` as a Kaggle
dataset, and `pip install` the wheels from it instead.

In [ ]:
%%bash
set -e
cd /kaggle/working

pip install -q --no-cache-dir \
  ninja cython timm shapely h5py scikit-image \
  opencv-python-headless "pycocotools>=2.0.7" \
  fvcore iopath cloudpickle omegaconf hydra-core black tabulate

pip install -q --no-cache-dir "git+https://github.com/cocodataset/panopticapi.git"
echo "deps ok"

In [ ]:
%%bash
set -e
cd /kaggle/working

export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6"
export FORCE_CUDA=1
export MAX_JOBS=2

rm -rf detectron2 d2repo          # kill the shadowing source dir first

if python -c "import detectron2; detectron2.__version__" 2>/dev/null; then
  echo "detectron2 already installed"
else
  git clone --depth 1 https://github.com/facebookresearch/detectron2.git d2repo
  pip install --no-build-isolation ./d2repo 2>&1 | tail -60
fi

cd /tmp && python -c "import detectron2; print('detectron2', detectron2.__version__)"

In [ ]:
%%bash
set -e
cd /kaggle/working

# Use your own fork here if you have local changes.
M2F_REPO=https://github.com/facebookresearch/Mask2Former.git

if [ ! -d Mask2Former ]; then
  git clone --depth 1 $M2F_REPO Mask2Former
fi
cd Mask2Former
pip install -q -r requirements.txt

# ---- patches for torch >= 1.11 / 2.x -------------------------------------
# THC/THCAtomics.cuh was deleted from pytorch; Tensor.type() is gone in torch 2.x.
cd mask2former/modeling/pixel_decoder/ops
sed -i 's|#include <THC/THCAtomics.cuh>|#include <ATen/cuda/Atomic.cuh>|' src/cuda/ms_deform_im2col_cuda.cuh
sed -i 's|\.type()\.is_cuda()|.is_cuda()|g' src/ms_deform_attn.h src/cuda/ms_deform_attn_cuda.cu src/cpu/ms_deform_attn_cpu.cpp
sed -i 's|value\.type()|value.scalar_type()|g' src/cuda/ms_deform_attn_cuda.cu
grep -n "Atomic.cuh" src/cuda/ms_deform_im2col_cuda.cuh || true

# ---- build the kernel ----------------------------------------------------
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6"
export FORCE_CUDA=1
export MAX_JOBS=4
pip install -q --no-build-isolation -e . || true
python -c "import MultiScaleDeformableAttention" 2>/dev/null || python setup.py build_ext --inplace
ls *.so 2>/dev/null || true
echo "MSDeformAttn built"

In [ ]:
import sys, importlib
sys.path.insert(0, "/kaggle/working/Mask2Former")
# last-resort path for a `build_ext --inplace` kernel
sys.path.append("/kaggle/working/Mask2Former/mask2former/modeling/pixel_decoder/ops")

import torch, detectron2
import MultiScaleDeformableAttention as MSDA
from mask2former import add_maskformer2_config
from detectron2.projects.deeplab import add_deeplab_config

print("detectron2", detectron2.__version__)
print("MSDeformAttn ok ->", MSDA.__file__)

# quick numerical smoke test of the kernel
from mask2former.modeling.pixel_decoder.ops.functions import MSDeformAttnFunction
print("kernel import ok")

## 2. Find the datasets

The three crop folders are YOLO-seg layout (`train/images`, `train/labels`, ...). The cell below
globs `/kaggle/input` for anything that looks like that and matches it to a crop by keyword, so it
works whether the mount is `/kaggle/input/smart-crops/...` or
`/kaggle/input/datasets/muhammadibnerafiq/smart-crops/...`.

If auto-detection picks the wrong folder, just hard-code `DATASETS[...]["root"]` in the next cell.

In [ ]:
import glob, os, json
from pathlib import Path

def discover_yolo_roots(search_root="/kaggle/input"):
    roots = set()
    for p in glob.glob(os.path.join(search_root, "**", "train", "images"), recursive=True):
        roots.add(str(Path(p).parent.parent))
    return sorted(roots)

ALL_ROOTS = discover_yolo_roots()
print("YOLO-style dataset roots found under /kaggle/input:")
for r in ALL_ROOTS:
    n = len(glob.glob(os.path.join(r, "train", "images", "*")))
    print(f"  {n:6d} train images   {r}")

In [ ]:
# keyword -> (dataset key, class name). Edit "root" below if auto-detect guessed wrong.
KEYWORDS = {
    "cabbage": ("cabbage", "cabbage"),
    "kale":    ("kale",    "purple_kale"),
    "onion":   ("onion",   "yellow_onion"),
}

def pick(keyword):
    hits = [r for r in ALL_ROOTS if keyword in r.lower()]
    if not hits:
        return None
    # deepest path wins (handles .../yellow_onion_extended/yellow_onion_extended)
    return max(hits, key=lambda r: (len(glob.glob(os.path.join(r, "train", "images", "*"))), len(r)))

DATASETS = {}
for kw, (key, cls) in KEYWORDS.items():
    root = pick(kw)
    DATASETS[key] = {"root": root, "names": [cls]}

for k, v in DATASETS.items():
    print(f"{k:8s} -> {v['root']}")

missing = [k for k, v in DATASETS.items() if v["root"] is None]
assert not missing, f"could not locate {missing}; set DATASETS[key]['root'] manually"

## 3. YOLO-seg labels -> COCO instance JSON

Each `labels/*.txt` line is `class x1 y1 x2 y2 ... xn yn` with normalised coordinates
(a 4-number line is treated as a `cx cy w h` box and turned into a rectangle polygon).
Images with no label file are kept with zero annotations; detectron2 drops them at train
time via `DATALOADER.FILTER_EMPTY_ANNOTATIONS`.

In [ ]:
import cv2
import numpy as np
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
COCO_ROOT = Path("/kaggle/working/coco")
COCO_ROOT.mkdir(parents=True, exist_ok=True)

def _poly_area(xs, ys):
    return 0.5 * abs(np.dot(xs, np.roll(ys, 1)) - np.dot(ys, np.roll(xs, 1)))

def parse_yolo_line(vals, W, H):
    """Return (flat_polygon, bbox_xywh, area) in absolute pixels, or None."""
    cls = int(float(vals[0]))
    c = [float(v) for v in vals[1:]]
    if len(c) == 4:                       # cx cy w h -> rectangle
        cx, cy, w, h = c
        x0, y0 = (cx - w / 2) * W, (cy - h / 2) * H
        x1, y1 = (cx + w / 2) * W, (cy + h / 2) * H
        xs = np.array([x0, x1, x1, x0]); ys = np.array([y0, y0, y1, y1])
    elif len(c) >= 6 and len(c) % 2 == 0:  # polygon
        pts = np.array(c, dtype=np.float64).reshape(-1, 2)
        xs, ys = pts[:, 0] * W, pts[:, 1] * H
    else:
        return None

    xs = np.clip(xs, 0, W - 1); ys = np.clip(ys, 0, H - 1)
    if len(xs) < 3:
        return None
    x0, y0, x1, y1 = xs.min(), ys.min(), xs.max(), ys.max()
    bw, bh = x1 - x0, y1 - y0
    area = _poly_area(xs, ys)
    if bw < 1 or bh < 1 or area < 1:
        return None
    poly = np.stack([xs, ys], axis=1).reshape(-1).tolist()
    return cls, [round(v, 2) for v in poly], [round(float(x0), 2), round(float(y0), 2),
                                              round(float(bw), 2), round(float(bh), 2)], float(area)

def yolo_split_to_coco(split_dir, class_names, out_json):
    split_dir = Path(split_dir)
    img_dir, lbl_dir = split_dir / "images", split_dir / "labels"
    images, annotations = [], []
    ann_id, n_skip = 1, 0

    files = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT])
    for img_id, ip in enumerate(files, start=1):
        try:
            with Image.open(ip) as im:
                W, H = im.size
        except Exception:
            n_skip += 1
            continue
        images.append({"id": img_id, "file_name": ip.name, "width": W, "height": H})

        lp = lbl_dir / (ip.stem + ".txt")
        if not lp.exists():
            continue
        for line in lp.read_text().splitlines():
            vals = line.split()
            if len(vals) < 5:
                continue
            out = parse_yolo_line(vals, W, H)
            if out is None:
                n_skip += 1
                continue
            cls, poly, bbox, area = out
            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": cls + 1,
                "segmentation": [poly], "bbox": bbox, "area": area, "iscrowd": 0,
            })
            ann_id += 1

    coco = {
        "info": {"description": str(split_dir)},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i + 1, "name": n, "supercategory": "crop"}
                       for i, n in enumerate(class_names)],
    }
    Path(out_json).parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w") as f:
        json.dump(coco, f)
    return len(images), len(annotations), n_skip

SPLITS = ["train", "valid", "test"]
REGISTRY = {}

for key, spec in DATASETS.items():
    REGISTRY[key] = {"names": spec["names"], "splits": {}}
    for split in SPLITS:
        sdir = Path(spec["root"]) / split
        if not (sdir / "images").is_dir():
            print(f"  [skip] {key}/{split} not present")
            continue
        out_json = COCO_ROOT / key / f"{split}.json"
        ni, na, ns = yolo_split_to_coco(sdir, spec["names"], out_json)
        REGISTRY[key]["splits"][split] = {"json": str(out_json), "images": str(sdir / "images"),
                                          "n_images": ni, "n_anns": na}
        print(f"  {key:8s} {split:6s} {ni:5d} images  {na:7d} instances  ({ns} skipped)")

with open(COCO_ROOT / "registry.json", "w") as f:
    json.dump(REGISTRY, f, indent=2)
print("\nwrote", COCO_ROOT / "registry.json")

In [ ]:
# sanity check: draw the converted polygons on 2 random training images
import random
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

def preview(key, split="train", n=2, seed=0):
    info = REGISTRY[key]["splits"][split]
    coco = COCO(info["json"])
    ids = sorted(coco.getImgIds())
    random.Random(seed).shuffle(ids)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    axes = np.atleast_1d(axes)
    for ax, iid in zip(axes, ids[:n]):
        rec = coco.loadImgs([iid])[0]
        img = cv2.cvtColor(cv2.imread(os.path.join(info["images"], rec["file_name"])), cv2.COLOR_BGR2RGB)
        anns = coco.loadAnns(coco.getAnnIds(imgIds=[iid]))
        for a in anns:
            p = np.array(a["segmentation"][0], dtype=np.int32).reshape(-1, 2)
            cv2.polylines(img, [p], True, (255, 0, 0), 2)
        ax.imshow(img); ax.set_title(f"{key}/{split} {rec['file_name']} - {len(anns)} inst"); ax.axis("off")
    plt.tight_layout(); plt.show()

for k in REGISTRY:
    preview(k)

## 4. Register the datasets with detectron2

`train_net.py` has no hook for custom datasets, so we drop a `register_crops.py` next to it and
a thin `train_crops.py` launcher that imports it before calling the original `main()`.

In [ ]:
%%writefile /kaggle/working/Mask2Former/register_crops.py
import json
from detectron2.data import MetadataCatalog
from detectron2.data.datasets import register_coco_instances

REGISTRY_PATH = "/kaggle/working/coco/registry.json"


def register_all():
    with open(REGISTRY_PATH) as f:
        registry = json.load(f)
    for key, spec in registry.items():
        for split, info in spec["splits"].items():
            name = f"crops_{key}_{split}"
            if name in MetadataCatalog.list():
                continue
            register_coco_instances(name, {}, info["json"], info["images"])
            MetadataCatalog.get(name).thing_classes = spec["names"]
    return registry


REGISTRY = register_all()

In [ ]:
%%writefile /kaggle/working/Mask2Former/train_crops.py
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.append(os.path.join(HERE, "mask2former", "modeling", "pixel_decoder", "ops"))

import register_crops  # noqa: F401  (registering happens on import)
from detectron2.engine import default_argument_parser, launch
from train_net import main

if __name__ == "__main__":
    args = default_argument_parser().parse_args()
    print("Command Line Args:", args)
    launch(
        main,
        args.num_gpus,
        num_machines=args.num_machines,
        machine_rank=args.machine_rank,
        dist_url=args.dist_url,
        args=(args,),
    )

In [ ]:
# COCO-pretrained Mask2Former R50 instance-segmentation weights (MODEL_ZOO.md)
!mkdir -p /kaggle/working/weights
!wget -q -nc -P /kaggle/working/weights \
  https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/instance/maskformer2_R50_bs16_50ep/model_final_3c8ec9.pkl
!ls -lh /kaggle/working/weights

PRETRAINED = "/kaggle/working/weights/model_final_3c8ec9.pkl"

# Swin-T variant if you want a stronger backbone (needs ~2x the memory):
# https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/instance/maskformer2_swin_tiny_bs16_50ep/model_final_86143f.pkl

## 5. Per-dataset config

Fine-tuning from COCO weights converges far faster than the YOLO runs - 50 epochs is plenty,
where YOLO needed 500 from a much smaller pretrain. Batch size 2 and `IMAGE_SIZE 640` fit a
single T4/P100; raise `IMS_PER_BATCH` on a bigger GPU and scale `BASE_LR` with it.

In [ ]:
M2F_DIR = "/kaggle/working/Mask2Former"
RUNS = "/kaggle/working/runs_m2f"
CFG_DIR = "/kaggle/working/cfgs"
os.makedirs(CFG_DIR, exist_ok=True)

BASE_CFG = f"{M2F_DIR}/configs/coco/instance-segmentation/maskformer2_R50_bs16_50ep.yaml"

EPOCHS = 50
IMS_PER_BATCH = 2
BASE_LR = 5e-5          # 1e-4 is the bs16 LR; we run bs2 and fine-tune, so lower
IMAGE_SIZE = 640
NUM_GPUS = 1            # keep at 1: register_crops runs in-process only for single-GPU launch

def make_cfg(key, epochs=EPOCHS, ims_per_batch=IMS_PER_BATCH, base_lr=BASE_LR,
             image_size=IMAGE_SIZE, weights=PRETRAINED):
    spec = REGISTRY[key]
    n_train = spec["splits"]["train"]["n_images"]
    n_cls = len(spec["names"])
    max_iter = max(1000, int(round(epochs * n_train / ims_per_batch)))
    steps = [int(max_iter * 0.75), int(max_iter * 0.9)]
    period = max(500, max_iter // 5)
    out_dir = f"{RUNS}/{key}"

    val_split = "valid" if "valid" in spec["splits"] else "test"
    yaml_text = f"""_BASE_: {BASE_CFG}
DATASETS:
  TRAIN: ("crops_{key}_train",)
  TEST: ("crops_{key}_{val_split}",)
MODEL:
  WEIGHTS: "{weights}"
  SEM_SEG_HEAD:
    NUM_CLASSES: {n_cls}
SOLVER:
  IMS_PER_BATCH: {ims_per_batch}
  BASE_LR: {base_lr}
  MAX_ITER: {max_iter}
  STEPS: [{steps[0]}, {steps[1]}]
  CHECKPOINT_PERIOD: {period}
  WARMUP_ITERS: 100
  AMP:
    ENABLED: True
INPUT:
  IMAGE_SIZE: {image_size}
  MIN_SCALE: 0.5
  MAX_SCALE: 1.5
  MIN_SIZE_TEST: {image_size}
  MAX_SIZE_TEST: {int(image_size * 1.6)}
TEST:
  EVAL_PERIOD: {period}
  DETECTIONS_PER_IMAGE: 100
DATALOADER:
  NUM_WORKERS: 2
  FILTER_EMPTY_ANNOTATIONS: True
SEED: 42
OUTPUT_DIR: "{out_dir}"
"""
    path = f"{CFG_DIR}/{key}.yaml"
    with open(path, "w") as f:
        f.write(yaml_text)
    print(f"{key}: {n_train} train images -> MAX_ITER={max_iter} (~{epochs} epochs), out={out_dir}")
    return path

CFGS = {k: make_cfg(k) for k in REGISTRY}

In [ ]:
import time
import torch

# ---------------------------------------------------------
# CUDA warm-up
# ---------------------------------------------------------

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    # Dummy CUDA operations to initialize CUDA context
    x = torch.randn(
        1024, 1024,
        device="cuda"
    )

    for _ in range(10):
        x = torch.matmul(x, x)

    torch.cuda.synchronize()

    del x
    torch.cuda.empty_cache()

    print("CUDA warm-up complete.")

else:
    print("CUDA not available. Running on CPU.")

## 7. Results

Recomputes COCO mask AP from the saved predictions so the table does not depend on scraping
the training log. `AP` is mAP@50-95 and `AP50` is mAP@50 - the same quantities
`yolo segment val` prints as `mAP50-95(M)` and `mAP50(M)`.

In [ ]:
# =========================================================
# MASK2FORMER: DIRECT INFERENCE TIMING + COCO EVALUATION
# Metrics:
#   Total inference time
#   Mean inference time
#   FPS
#   AP
#   AP50
#   AP75
#   APl
#
# IMPORTANT:
# - Direct model inference timing
# - ONE inference pass only
# - No second timing/evaluation inference pass
# =========================================================

import os
import sys
import time
import torch

# =========================================================
# Mask2Former paths
# =========================================================

M2F_DIR = "/kaggle/working/Mask2Former"

if M2F_DIR not in sys.path:
    sys.path.insert(0, M2F_DIR)
    sys.path.append(
        os.path.join(
            M2F_DIR,
            "mask2former",
            "modeling",
            "pixel_decoder",
            "ops"
        )
    )

# =========================================================
# Imports
# =========================================================

import register_crops

from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from mask2former import add_maskformer2_config

from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import COCOEvaluator


# =========================================================
# Crop configurations
# =========================================================

crops = [
    {
        "name": "kale",
        "dataset": "crops_kale_test",
        "config": "/kaggle/working/cfgs/kale.yaml",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/mask2former/kale_model_final.pth",
    },
    {
        "name": "cabbage",
        "dataset": "crops_cabbage_test",
        "config": "/kaggle/working/cfgs/cabbage.yaml",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/mask2former/cabbage_model_final.pth",
    },
    {
        "name": "onion",
        "dataset": "crops_onion_test",
        "config": "/kaggle/working/cfgs/onion.yaml",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/mask2former/onion_model_final.pth",
    },
]


# =========================================================
# Device
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# =========================================================
# Store results
# =========================================================

benchmark_results = []


# =========================================================
# Run each crop
# =========================================================

for crop in crops:

    print("\n")
    print("=" * 90)
    print(f"PROCESSING: {crop['name'].upper()}")
    print("=" * 90)

    # =====================================================
    # Configuration
    # =====================================================

    cfg = get_cfg()

    add_deeplab_config(cfg)
    add_maskformer2_config(cfg)

    cfg.merge_from_file(crop["config"])

    cfg.MODEL.WEIGHTS = crop["weights"]
    cfg.DATASETS.TEST = (crop["dataset"],)
    cfg.MODEL.DEVICE = device

    cfg.freeze()

    # =====================================================
    # Build model
    # =====================================================

    model = build_model(cfg)

    DetectionCheckpointer(model).load(
        cfg.MODEL.WEIGHTS
    )

    model.eval()

    print("Model loaded.")

    # =====================================================
    # Build test loader
    # =====================================================

    test_loader = build_detection_test_loader(
        cfg,
        crop["dataset"]
    )

    dataset_size = len(test_loader.dataset)

    print(f"Test images: {dataset_size}")

    # =====================================================
    # Output directory
    # =====================================================

    output_dir = os.path.join(
        cfg.OUTPUT_DIR,
        "test_results",
        crop["name"]
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    # =====================================================
    # COCO evaluator
    # =====================================================

    evaluator = COCOEvaluator(
        crop["dataset"],
        tasks=("segm", "bbox"),
        distributed=False,
        output_dir=output_dir,
    )

    evaluator.reset()

    # =====================================================
    # DIRECT MODEL INFERENCE + TIMING
    #
    # ONE PASS ONLY
    # =====================================================

    print("\nRunning direct model inference...")

    total_time = 0.0
    processed_images = 0

    with torch.no_grad():

        for inputs in test_loader:

            # Number of images in this batch
            batch_size = len(inputs)

            # -------------------------------------------------
            # CUDA synchronization BEFORE model inference
            # -------------------------------------------------

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start_time = time.perf_counter()

            # -------------------------------------------------
            # DIRECT MODEL INFERENCE
            # -------------------------------------------------

            outputs = model(inputs)

            # -------------------------------------------------
            # CUDA synchronization AFTER model inference
            # -------------------------------------------------

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end_time = time.perf_counter()

            # -------------------------------------------------
            # Add ONLY model inference time
            # -------------------------------------------------

            total_time += end_time - start_time
            processed_images += batch_size

            # -------------------------------------------------
            # Send predictions to COCO evaluator
            #
            # This does NOT run the model again.
            # -------------------------------------------------

            evaluator.process(
                inputs,
                outputs
            )

    # =====================================================
    # COCO EVALUATION
    #
    # This evaluates the predictions already generated above.
    # No second inference pass.
    # =====================================================

    print("\nRunning COCO evaluation...")

    results = evaluator.evaluate()

    # =====================================================
    # Timing
    # =====================================================

    mean_time = (
        total_time / processed_images
        if processed_images > 0
        else 0.0
    )

    fps = (
        processed_images / total_time
        if total_time > 0
        else 0.0
    )

    # =====================================================
    # Extract SEGMENTATION metrics
    # =====================================================

    segm_results = results.get("segm", {})

    ap = segm_results.get("AP", float("nan"))
    ap50 = segm_results.get("AP50", float("nan"))
    ap75 = segm_results.get("AP75", float("nan"))
    apl = segm_results.get("APl", float("nan"))

    # =====================================================
    # Print results
    # =====================================================

    print("\n")
    print("=" * 80)
    print(f"{crop['name'].upper()} RESULTS")
    print("=" * 80)

    print("\nInference Performance")
    print("-" * 60)

    print(f"Images               : {processed_images}")
    print(f"Total inference time : {total_time:.4f} sec")
    print(f"Mean inference time  : {mean_time:.6f} sec/image")
    print(f"FPS                  : {fps:.2f}")

    print("\nSegmentation COCO Metrics")
    print("-" * 60)

    print(f"AP                   : {ap:.2f}")
    print(f"AP50                 : {ap50:.2f}")
    print(f"AP75                 : {ap75:.2f}")
    print(f"APl                  : {apl:.2f}")

    # =====================================================
    # Store results
    # =====================================================

    benchmark_results.append({
        "crop": crop["name"],
        "images": processed_images,

        "total_time_sec": total_time,
        "mean_time_sec": mean_time,
        "fps": fps,

        "AP": ap,
        "AP50": ap50,
        "AP75": ap75,
        "APl": apl,
    })

    # =====================================================
    # Free GPU memory
    # =====================================================

    del model
    del test_loader
    del evaluator

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n\n")
print("=" * 110)
print("FINAL MASK2FORMER RESULTS")
print("=" * 110)

print(
    f"{'Crop':<12}"
    f"{'Images':<10}"
    f"{'Total (s)':<15}"
    f"{'Mean (s)':<15}"
    f"{'FPS':<10}"
    f"{'AP':<10}"
    f"{'AP50':<10}"
    f"{'AP75':<10}"
    f"{'APl':<10}"
)

print("-" * 110)

for result in benchmark_results:

    print(
        f"{result['crop'].upper():<12}"
        f"{result['images']:<10}"
        f"{result['total_time_sec']:<15.4f}"
        f"{result['mean_time_sec']:<15.6f}"
        f"{result['fps']:<10.2f}"
        f"{result['AP']:<10.2f}"
        f"{result['AP50']:<10.2f}"
        f"{result['AP75']:<10.2f}"
        f"{result['APl']:<10.2f}"
    )

In [ ]:
!pip install ultralytics

In [ ]:
# =========================================================
# YOLO SEGMENTATION: DIRECT INFERENCE TIMING + COCO EVALUATION
#
# Metrics:
#   Total inference time
#   Mean inference time
#   FPS
#   AP
#   AP50
#   AP75
#   APl
#
# IMPORTANT:
# - Direct model inference timing
# - ONE inference pass only
# - No second inference/evaluation inference pass
# =========================================================

import os
import json
import time

import cv2
import numpy as np
import torch

import pycocotools.mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from ultralytics import YOLO


# =========================================================
# YOLO MODEL CONFIGURATIONS
# =========================================================

models = [
    {
        "name": "kale",
        "key": "kale",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/yolov12s_weights/yolo_weights/kale_best.pt",
    },
    {
        "name": "cabbage",
        "key": "cabbage",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/yolov12s_weights/yolo_weights/cabbage_best.pt",
    },
    {
        "name": "onion",
        "key": "onion",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/yolov12s_weights/yolo_weights/yello_onion_best.pt",
    },
]


# =========================================================
# SETTINGS
# =========================================================

registry_path = "/kaggle/working/coco/registry.json"

split = "test"

CONF = 0.001
IOU = 0.7
MAX_DET = 100


# =========================================================
# DEVICE
# =========================================================

device = 0 if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("YOLO SEGMENTATION EVALUATION + INFERENCE BENCHMARK")
print("=" * 80)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")


# =========================================================
# LOAD REGISTRY ONCE
# =========================================================

with open(registry_path, "r") as f:
    registry = json.load(f)


# =========================================================
# STORE RESULTS
# =========================================================

all_results = []


# =========================================================
# PROCESS EACH MODEL
# =========================================================

for model_info in models:

    name = model_info["name"]
    key = model_info["key"]
    weights = model_info["weights"]

    print("\n")
    print("=" * 80)
    print(f"PROCESSING: {name.upper()}")
    print("=" * 80)

    # =====================================================
    # LOAD COCO GROUND TRUTH
    # =====================================================

    gt_json = registry[key]["splits"][split]["json"]
    img_dir = registry[key]["splits"][split]["images"]

    coco_gt = COCO(gt_json)

    img_ids = coco_gt.getImgIds()

    print(f"Test images: {len(img_ids)}")

    # =====================================================
    # LOAD YOLO MODEL
    # =====================================================

    print("Loading model...")

    model = YOLO(weights)

    print("Model loaded.")

    # =====================================================
    # STORAGE FOR COCO PREDICTIONS
    # =====================================================

    coco_predictions = []

    # =====================================================
    # INFERENCE TIMING
    # =====================================================

    inference_times = []

    print("\nRunning inference...")

    for img_id in img_ids:

        # -------------------------------------------------
        # Image information
        # -------------------------------------------------

        info = coco_gt.loadImgs([img_id])[0]

        h = info["height"]
        w = info["width"]

        path = os.path.join(
            img_dir,
            info["file_name"]
        )

        # -------------------------------------------------
        # Load image BEFORE timing
        # -------------------------------------------------

        image = cv2.imread(path)

        if image is None:
            print(
                f"Warning: could not read {path}"
            )
            continue

        # -------------------------------------------------
        # Synchronize CUDA BEFORE timing
        # -------------------------------------------------

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        # =================================================
        # ACTUAL YOLO INFERENCE
        # =================================================

        preds = model.predict(
            image,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            retina_masks=True,
            verbose=False,
            device=device,
        )[0]

        # -------------------------------------------------
        # Synchronize CUDA AFTER inference
        # -------------------------------------------------

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        # -------------------------------------------------
        # Store ONLY model inference time
        # -------------------------------------------------

        elapsed = end_time - start_time

        inference_times.append(elapsed)

        # =================================================
        # NO PREDICTIONS
        # =================================================

        if (
            preds.masks is None
            or len(preds.boxes) == 0
        ):
            continue

        # =================================================
        # EXTRACT PREDICTIONS
        # =================================================

        boxes = (
            preds.boxes.xyxy
            .cpu()
            .numpy()
        )

        confs = (
            preds.boxes.conf
            .cpu()
            .numpy()
        )

        classes = (
            preds.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        masks = (
            preds.masks.data
            .cpu()
            .numpy()
        )

        # =================================================
        # CONVERT TO COCO FORMAT
        # =================================================

        for i in range(len(boxes)):

            x1, y1, x2, y2 = boxes[i]

            bbox = [
                round(float(x1), 2),
                round(float(y1), 2),
                round(float(x2 - x1), 2),
                round(float(y2 - y1), 2),
            ]

            # -------------------------------------------------
            # Mask
            # -------------------------------------------------

            m = masks[i]

            if m.shape != (h, w):

                m = cv2.resize(
                    m,
                    (w, h),
                    interpolation=cv2.INTER_NEAREST
                )

            m = (
                m > 0.5
            ).astype(np.uint8)

            # -------------------------------------------------
            # RLE encoding
            # -------------------------------------------------

            rle = mask_utils.encode(
                np.asfortranarray(m)
            )

            rle["counts"] = (
                rle["counts"]
                .decode("utf-8")
            )

            # -------------------------------------------------
            # COCO prediction
            # -------------------------------------------------

            coco_predictions.append({
                "image_id": img_id,

                # YOLO class 0 -> COCO category 1
                "category_id": (
                    int(classes[i]) + 1
                ),

                "bbox": bbox,

                "score": round(
                    float(confs[i]),
                    4
                ),

                "segmentation": rle,
            })


    # =====================================================
    # INFERENCE STATISTICS
    # =====================================================

    num_images = len(inference_times)

    total_time = sum(inference_times)

    mean_time = (
        total_time / num_images
        if num_images > 0
        else 0.0
    )

    fps = (
        num_images / total_time
        if total_time > 0
        else 0.0
    )


    # =====================================================
    # COCO SEGMENTATION EVALUATION
    # =====================================================

    print("\n")
    print("-" * 70)
    print("COCO SEGMENTATION EVALUATION")
    print("-" * 70)

    if len(coco_predictions) == 0:

        print(
            "No predictions generated. "
            "Skipping COCO evaluation."
        )

        metrics = {
            "AP": 0.0,
            "AP50": 0.0,
            "AP75": 0.0,
            "APl": 0.0,
        }

    else:

        coco_dt = coco_gt.loadRes(
            coco_predictions
        )

        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            iouType="segm"
        )

        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

        stats = evaluator.stats

        metrics = {
            "AP": round(
                stats[0] * 100,
                2
            ),

            "AP50": round(
                stats[1] * 100,
                2
            ),

            "AP75": round(
                stats[2] * 100,
                2
            ),

            "APl": round(
                stats[5] * 100,
                2
            ),
        }


    # =====================================================
    # STORE RESULTS
    # =====================================================

    all_results.append({

        "model": f"YOLO-Seg ({name})",

        "crop": name,

        # Metrics
        "AP": metrics["AP"],
        "AP50": metrics["AP50"],
        "AP75": metrics["AP75"],
        "APl": metrics["APl"],

        # Inference
        "total_time_sec": total_time,
        "mean_time_sec": mean_time,
        "FPS": fps,
    })


    # =====================================================
    # CLEANUP
    # =====================================================

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================================================
# FINAL YOLO-Seg RESULTS
# =========================================================

print("\n\n")
print("=" * 105)
print("FINAL YOLO-Seg RESULTS")
print("=" * 105)

print(
    f"{'Crop':<12}"
    f"{'Total Inference Time (s)':<25}"
    f"{'Mean Inference Time (s)':<25}"
    f"{'FPS':<10}"
    f"{'AP':<10}"
    f"{'AP50':<10}"
    f"{'AP75':<10}"
    f"{'APl':<10}"
)

print("-" * 105)

for result in all_results:

    print(
        f"{result['crop'].upper():<12}"
        f"{result['total_time_sec']:<25.4f}"
        f"{result['mean_time_sec']:<25.6f}"
        f"{result['FPS']:<10.2f}"
        f"{result['AP']:<10.2f}"
        f"{result['AP50']:<10.2f}"
        f"{result['AP75']:<10.2f}"
        f"{result['APl']:<10.2f}"
    )

In [ ]:
# =========================================================
# YOLO SEGMENTATION: DIRECT INFERENCE TIMING + COCO EVALUATION
#
# Metrics:
#   Total inference time
#   Mean inference time
#   FPS
#   AP
#   AP50
#   AP75
#   APl
#
# IMPORTANT:
# - Direct model inference timing
# - ONE inference pass only
# - No second inference/evaluation inference pass
# =========================================================

import os
import json
import time

import cv2
import numpy as np
import torch

import pycocotools.mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from ultralytics import YOLO


# =========================================================
# YOLO MODEL CONFIGURATIONS
# =========================================================

models = [
    {
        "name": "kale",
        "key": "kale",
        "weights": "/kaggle/input/datasets/divija4505/yolo26s-weights/kale_best.pt",
    },
    {
        "name": "cabbage",
        "key": "cabbage",
        "weights": "/kaggle/input/datasets/divija4505/yolo26s-weights/cabbage_best.pt",
    },
    {
        "name": "onion",
        "key": "onion",
        "weights": "/kaggle/input/datasets/divija4505/yolo26s-weights/onion_best.pt",
    },
]


# =========================================================
# SETTINGS
# =========================================================

registry_path = "/kaggle/working/coco/registry.json"

split = "test"

CONF = 0.001
IOU = 0.7
MAX_DET = 100


# =========================================================
# DEVICE
# =========================================================

device = 0 if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("YOLO SEGMENTATION EVALUATION + INFERENCE BENCHMARK")
print("=" * 80)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")


# =========================================================
# LOAD REGISTRY ONCE
# =========================================================

with open(registry_path, "r") as f:
    registry = json.load(f)


# =========================================================
# STORE RESULTS
# =========================================================

all_results = []


# =========================================================
# PROCESS EACH MODEL
# =========================================================

for model_info in models:

    name = model_info["name"]
    key = model_info["key"]
    weights = model_info["weights"]

    print("\n")
    print("=" * 80)
    print(f"PROCESSING: {name.upper()}")
    print("=" * 80)

    # =====================================================
    # LOAD COCO GROUND TRUTH
    # =====================================================

    gt_json = registry[key]["splits"][split]["json"]
    img_dir = registry[key]["splits"][split]["images"]

    coco_gt = COCO(gt_json)

    img_ids = coco_gt.getImgIds()

    print(f"Test images: {len(img_ids)}")

    # =====================================================
    # LOAD YOLO MODEL
    # =====================================================

    print("Loading model...")

    model = YOLO(weights)

    print("Model loaded.")

    # =====================================================
    # STORAGE FOR COCO PREDICTIONS
    # =====================================================

    coco_predictions = []

    # =====================================================
    # INFERENCE TIMING
    # =====================================================

    inference_times = []

    print("\nRunning inference...")

    for img_id in img_ids:

        # -------------------------------------------------
        # Image information
        # -------------------------------------------------

        info = coco_gt.loadImgs([img_id])[0]

        h = info["height"]
        w = info["width"]

        path = os.path.join(
            img_dir,
            info["file_name"]
        )

        # -------------------------------------------------
        # Load image BEFORE timing
        # -------------------------------------------------

        image = cv2.imread(path)

        if image is None:
            print(
                f"Warning: could not read {path}"
            )
            continue

        # -------------------------------------------------
        # Synchronize CUDA BEFORE timing
        # -------------------------------------------------

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        # =================================================
        # ACTUAL YOLO INFERENCE
        # =================================================

        preds = model.predict(
            image,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            retina_masks=True,
            verbose=False,
            device=device,
        )[0]

        # -------------------------------------------------
        # Synchronize CUDA AFTER inference
        # -------------------------------------------------

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        # -------------------------------------------------
        # Store ONLY model inference time
        # -------------------------------------------------

        elapsed = end_time - start_time

        inference_times.append(elapsed)

        # =================================================
        # NO PREDICTIONS
        # =================================================

        if (
            preds.masks is None
            or len(preds.boxes) == 0
        ):
            continue

        # =================================================
        # EXTRACT PREDICTIONS
        # =================================================

        boxes = (
            preds.boxes.xyxy
            .cpu()
            .numpy()
        )

        confs = (
            preds.boxes.conf
            .cpu()
            .numpy()
        )

        classes = (
            preds.boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )

        masks = (
            preds.masks.data
            .cpu()
            .numpy()
        )

        # =================================================
        # CONVERT TO COCO FORMAT
        # =================================================

        for i in range(len(boxes)):

            x1, y1, x2, y2 = boxes[i]

            bbox = [
                round(float(x1), 2),
                round(float(y1), 2),
                round(float(x2 - x1), 2),
                round(float(y2 - y1), 2),
            ]

            # -------------------------------------------------
            # Mask
            # -------------------------------------------------

            m = masks[i]

            if m.shape != (h, w):

                m = cv2.resize(
                    m,
                    (w, h),
                    interpolation=cv2.INTER_NEAREST
                )

            m = (
                m > 0.5
            ).astype(np.uint8)

            # -------------------------------------------------
            # RLE encoding
            # -------------------------------------------------

            rle = mask_utils.encode(
                np.asfortranarray(m)
            )

            rle["counts"] = (
                rle["counts"]
                .decode("utf-8")
            )

            # -------------------------------------------------
            # COCO prediction
            # -------------------------------------------------

            coco_predictions.append({
                "image_id": img_id,

                # YOLO class 0 -> COCO category 1
                "category_id": (
                    int(classes[i]) + 1
                ),

                "bbox": bbox,

                "score": round(
                    float(confs[i]),
                    4
                ),

                "segmentation": rle,
            })


    # =====================================================
    # INFERENCE STATISTICS
    # =====================================================

    num_images = len(inference_times)

    total_time = sum(inference_times)

    mean_time = (
        total_time / num_images
        if num_images > 0
        else 0.0
    )

    fps = (
        num_images / total_time
        if total_time > 0
        else 0.0
    )


    # =====================================================
    # COCO SEGMENTATION EVALUATION
    # =====================================================

    print("\n")
    print("-" * 70)
    print("COCO SEGMENTATION EVALUATION")
    print("-" * 70)

    if len(coco_predictions) == 0:

        print(
            "No predictions generated. "
            "Skipping COCO evaluation."
        )

        metrics = {
            "AP": 0.0,
            "AP50": 0.0,
            "AP75": 0.0,
            "APl": 0.0,
        }

    else:

        coco_dt = coco_gt.loadRes(
            coco_predictions
        )

        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            iouType="segm"
        )

        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

        stats = evaluator.stats

        metrics = {
            "AP": round(
                stats[0] * 100,
                2
            ),

            "AP50": round(
                stats[1] * 100,
                2
            ),

            "AP75": round(
                stats[2] * 100,
                2
            ),

            "APl": round(
                stats[5] * 100,
                2
            ),
        }


    # =====================================================
    # STORE RESULTS
    # =====================================================

    all_results.append({

        "model": f"YOLO-Seg ({name})",

        "crop": name,

        # Metrics
        "AP": metrics["AP"],
        "AP50": metrics["AP50"],
        "AP75": metrics["AP75"],
        "APl": metrics["APl"],

        # Inference
        "total_time_sec": total_time,
        "mean_time_sec": mean_time,
        "FPS": fps,
    })


    # =====================================================
    # CLEANUP
    # =====================================================

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================================================
# FINAL YOLO-Seg RESULTS
# =========================================================

print("\n\n")
print("=" * 105)
print("FINAL YOLO-Seg RESULTS")
print("=" * 105)

print(
    f"{'Crop':<12}"
    f"{'Total Inference Time (s)':<25}"
    f"{'Mean Inference Time (s)':<25}"
    f"{'FPS':<10}"
    f"{'AP':<10}"
    f"{'AP50':<10}"
    f"{'AP75':<10}"
    f"{'APl':<10}"
)

print("-" * 105)

for result in all_results:

    print(
        f"{result['crop'].upper():<12}"
        f"{result['total_time_sec']:<25.4f}"
        f"{result['mean_time_sec']:<25.6f}"
        f"{result['FPS']:<10.2f}"
        f"{result['AP']:<10.2f}"
        f"{result['AP50']:<10.2f}"
        f"{result['AP75']:<10.2f}"
        f"{result['APl']:<10.2f}"
    )

In [ ]:
import os
import time
import statistics
import torch

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset


# ============================================================
# 1. CONFIGURE MASK R-CNN
# ============================================================

cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

# Test dataset
cfg.DATASETS.TEST = ("onion_test",)

# One class: onion plant
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

# ------------------------------------------------------------
# IMPORTANT:
# Put your TRAINED Mask R-CNN checkpoint here
# ------------------------------------------------------------
cfg.MODEL.WEIGHTS = "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/Mask R CNN weights/onion_model_final.pth"

# Device
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Detection threshold
# Keep this low for COCO evaluation
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.001

print("Device:", cfg.MODEL.DEVICE)
print("Weights:", cfg.MODEL.WEIGHTS)


# ============================================================
# 2. LOAD TRAINED MASK R-CNN
# ============================================================

predictor = DefaultPredictor(cfg)

model = predictor.model
model.eval()

print("\nMask R-CNN loaded successfully.")

In [ ]:
# ============================================================
# MASK R-CNN: EVALUATE ALL CROPS
# COCO EVALUATION + INFERENCE TIMING
# ============================================================

import os
import time
import statistics
import cv2
import torch

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.data import DatasetCatalog, build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset


# ============================================================
# 1. CROPS + TRAINED WEIGHTS
# ============================================================

CROPS = {
    "kale": {
        "dataset": "crops_kale_test",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/Mask R CNN weights/kale_model_final.pth",
    },

    "cabbage": {
        "dataset": "crops_cabbage_test",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/Mask R CNN weights/cabbage_model_final.pth",
    },

    "onion": {
        "dataset": "crops_onion_test",
        "weights": "/kaggle/input/datasets/muhammadibnerafiq/smart-crops/Mask R CNN weights/onion_model_final.pth",
    },
}


# ============================================================
# 2. SETTINGS
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 1

# Low threshold for COCO evaluation
SCORE_THRESH = 0.001

OUTPUT_ROOT = "/kaggle/working/mask_rcnn_evaluation"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Device:", DEVICE)


# ============================================================
# 3. STORE FINAL RESULTS
# ============================================================

all_results = {}


# ============================================================
# 4. LOOP OVER CROPS
# ============================================================

for crop_name, crop_info in CROPS.items():

    dataset_name = crop_info["dataset"]
    weights_path = crop_info["weights"]

    print("\n")
    print("=" * 80)
    print(f"                         {crop_name.upper()}")
    print("=" * 80)

    print(f"Dataset : {dataset_name}")
    print(f"Weights : {weights_path}")

    # --------------------------------------------------------
    # Check weights
    # --------------------------------------------------------

    if not os.path.exists(weights_path):
        print(f"WARNING: Weights not found: {weights_path}")
        print("Skipping this crop.")
        continue

    # --------------------------------------------------------
    # Check dataset registration
    # --------------------------------------------------------

    if dataset_name not in DatasetCatalog.list():
        print(f"WARNING: Dataset not registered: {dataset_name}")
        print("Skipping this crop.")
        continue

    # ========================================================
    # CREATE CONFIG
    # ========================================================

    cfg = get_cfg()

    cfg.merge_from_file(
        model_zoo.get_config_file(
            "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
        )
    )

    cfg.DATASETS.TEST = (dataset_name,)

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES

    cfg.MODEL.WEIGHTS = weights_path

    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH

    cfg.MODEL.DEVICE = DEVICE

    # ========================================================
    # LOAD TRAINED MODEL
    # ========================================================

    predictor = DefaultPredictor(cfg)

    model = predictor.model
    model.eval()

    print("\nModel loaded.")


    # ========================================================
    # COCO EVALUATION
    # ========================================================

    test_loader = build_detection_test_loader(
        cfg,
        dataset_name
    )

    output_dir = os.path.join(
        OUTPUT_ROOT,
        crop_name
    )

    os.makedirs(output_dir, exist_ok=True)

    evaluator = COCOEvaluator(
        dataset_name,
        tasks=("segm", "bbox"),
        output_dir=output_dir
    )

    print("\nRunning COCO evaluation...")

    results = inference_on_dataset(
        model,
        test_loader,
        evaluator
    )


    # ========================================================
    # EXTRACT BBOX COCO METRICS
    # ========================================================

    bbox_results = results.get("bbox", {})

    bbox_ap = bbox_results.get("AP", float("nan"))
    bbox_ap50 = bbox_results.get("AP50", float("nan"))
    bbox_ap75 = bbox_results.get("AP75", float("nan"))
    bbox_aps = bbox_results.get("APs", float("nan"))
    bbox_apm = bbox_results.get("APm", float("nan"))
    bbox_apl = bbox_results.get("APl", float("nan"))


    # ========================================================
    # EXTRACT SEGMENTATION COCO METRICS
    # ========================================================

    segm_results = results.get("segm", {})

    segm_ap = segm_results.get("AP", float("nan"))
    segm_ap50 = segm_results.get("AP50", float("nan"))
    segm_ap75 = segm_results.get("AP75", float("nan"))
    segm_aps = segm_results.get("APs", float("nan"))
    segm_apm = segm_results.get("APm", float("nan"))
    segm_apl = segm_results.get("APl", float("nan"))


    # ========================================================
    # INFERENCE TIMING
    # ========================================================

    print("\nRunning inference timing...")

    dataset_dicts = DatasetCatalog.get(dataset_name)

    inference_times = []

    with torch.no_grad():

        for idx, d in enumerate(dataset_dicts):

            # ------------------------------------------------
            # Load image BEFORE timing
            # ------------------------------------------------

            image = cv2.imread(d["file_name"])

            if image is None:
                print(
                    f"WARNING: Could not read image: "
                    f"{d['file_name']}"
                )
                continue

            # ------------------------------------------------
            # Synchronize GPU BEFORE timing
            # ------------------------------------------------

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start_time = time.perf_counter()

            # ------------------------------------------------
            # Mask R-CNN inference
            # ------------------------------------------------

            outputs = model([
                {
                    "image": torch.as_tensor(
                        image.astype("float32").transpose(2, 0, 1)
                    ),
                    "height": image.shape[0],
                    "width": image.shape[1],
                }
            ])

            # ------------------------------------------------
            # Synchronize GPU AFTER inference
            # ------------------------------------------------

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end_time = time.perf_counter()

            elapsed = end_time - start_time

            inference_times.append(elapsed)

            if (idx + 1) % 100 == 0:
                print(
                    f"Processed "
                    f"{idx + 1}/{len(dataset_dicts)}"
                )


    # ========================================================
    # TIMING STATISTICS
    # ========================================================

    num_images = len(inference_times)

    total_time = sum(inference_times)

    mean_time = statistics.mean(inference_times)

    median_time = statistics.median(inference_times)

    std_time = (
        statistics.stdev(inference_times)
        if num_images > 1
        else 0.0
    )

    min_time = min(inference_times)

    max_time = max(inference_times)

    fps = (
        num_images / total_time
        if total_time > 0
        else 0.0
    )


    # ========================================================
    # SAVE ALL RESULTS
    # ========================================================

    all_results[crop_name] = {

        # ---------------- BBOX ----------------

        "bbox_AP": bbox_ap,
        "bbox_AP50": bbox_ap50,
        "bbox_AP75": bbox_ap75,
        "bbox_APs": bbox_aps,
        "bbox_APm": bbox_apm,
        "bbox_APl": bbox_apl,

        # ---------------- SEGMENTATION ----------------

        "segm_AP": segm_ap,
        "segm_AP50": segm_ap50,
        "segm_AP75": segm_ap75,
        "segm_APs": segm_aps,
        "segm_APm": segm_apm,
        "segm_APl": segm_apl,

        # ---------------- INFERENCE ----------------

        "images": num_images,
        "total_time": total_time,
        "mean_latency": mean_time,
        "median_latency": median_time,
        "std_latency": std_time,
        "min_latency": min_time,
        "max_latency": max_time,
        "FPS": fps,
    }


    # ========================================================
    # PRINT CROP RESULTS
    # ========================================================

    print("\n")
    print("=" * 80)
    print(f"{crop_name.upper()} RESULTS")
    print("=" * 80)

    # --------------------------------------------------------
    # BBOX
    # --------------------------------------------------------

    print("\nBBOX COCO:")
    print(f"  AP       : {bbox_ap:.2f}")
    print(f"  AP50     : {bbox_ap50:.2f}")
    print(f"  AP75     : {bbox_ap75:.2f}")
    print(f"  APs      : {bbox_aps:.2f}")
    print(f"  APm      : {bbox_apm:.2f}")
    print(f"  APl      : {bbox_apl:.2f}")

    # --------------------------------------------------------
    # SEGMENTATION
    # --------------------------------------------------------

    print("\nSEGMENTATION COCO:")
    print(f"  AP       : {segm_ap:.2f}")
    print(f"  AP50     : {segm_ap50:.2f}")
    print(f"  AP75     : {segm_ap75:.2f}")
    print(f"  APs      : {segm_aps:.2f}")
    print(f"  APm      : {segm_apm:.2f}")
    print(f"  APl      : {segm_apl:.2f}")

    # --------------------------------------------------------
    # INFERENCE
    # --------------------------------------------------------

    print("\nINFERENCE:")
    print(f"  Images           : {num_images}")
    print(f"  Total time       : {total_time:.4f} sec")
    print(f"  Mean latency     : {mean_time:.4f} sec/image")
    print(f"  Median latency   : {median_time:.4f} sec/image")
    print(f"  Std latency      : {std_time:.4f} sec")
    print(f"  Min latency      : {min_time:.4f} sec")
    print(f"  Max latency      : {max_time:.4f} sec")
    print(f"  FPS              : {fps:.2f}")


# ============================================================
# 5. FINAL SEGMENTATION + INFERENCE SUMMARY
# ============================================================

print("\n\n")
print("=" * 180)
print("                     FINAL MASK R-CNN RESULTS")
print("=" * 180)

print(
    f"{'Crop':<10}"
    f"{'AP':>10}"
    f"{'AP50':>10}"
    f"{'AP75':>10}"
    f"{'APs':>10}"
    f"{'APm':>10}"
    f"{'APl':>10}"
    f"{'Total(s)':>14}"
    f"{'Mean(s)':>14}"
    f"{'Median(s)':>14}"
    f"{'Std(s)':>12}"
    f"{'Min(s)':>12}"
    f"{'Max(s)':>12}"
    f"{'FPS':>10}"
)

print("-" * 180)

for crop, r in all_results.items():

    print(
        f"{crop:<10}"
        f"{r['segm_AP']:>10.2f}"
        f"{r['segm_AP50']:>10.2f}"
        f"{r['segm_AP75']:>10.2f}"
        f"{r['segm_APs']:>10.2f}"
        f"{r['segm_APm']:>10.2f}"
        f"{r['segm_APl']:>10.2f}"
        f"{r['total_time']:>14.4f}"
        f"{r['mean_latency']:>14.4f}"
        f"{r['median_latency']:>14.4f}"
        f"{r['std_latency']:>12.4f}"
        f"{r['min_latency']:>12.4f}"
        f"{r['max_latency']:>12.4f}"
        f"{r['FPS']:>10.2f}"
    )

